# ACT: chunks, CVAE, and temporal ensembling

**Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware** · Zhao et al. · 2023

[Original paper](https://arxiv.org/abs/2304.13705)

**Learning goal:** Build a tiny conditional VAE for action chunks, separate posterior training from inference, and ensemble overlapping predictions.

## Before starting

Introductory VLA, action chunks, Gaussian latents, and KL.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

ACT predicts future action chunks rather than a single next action. Its CVAE models demonstration variability: the training posterior sees target actions, while the decoder uses a latent and observation. Target actions are unavailable at inference.

Temporal ensembling combines predictions for the same absolute time made at different times, not arbitrary chunk positions. A state-only MLP replaces the visual Transformer so we can inspect the objective and indexing on CPU.

## Reproduction boundary

Educational state-only CVAE and same-time ensembling; not the official ACT visual Transformer, LeRobot trainer, or a real-robot validation.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`z = mu + exp(0.5*logvar)*epsilon`; `KL(q||N(0,I)) = 0.5*sum(mu² + exp(logvar) - 1 - logvar)`.

`loss = masked_L1(actions_hat, actions) + beta*KL`. Average reconstruction over valid coordinates; sum latent dimensions before averaging KL over the batch. Inference here uses z=0. Our ensemble uses `weight=exp(-k*age)`, with larger age for older chunks: an explicit recency-weighted convention, not a claim about the original code’s exact weighting convention.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

The posterior receives state and the demonstration chunk; inference decoding uses only state and z=0.

In [ ]:
B, K, A, Z = 48, 4, 2, 3
state = torch.randn(B, 3)
actions = state[:, None, :2] * 0.3 + torch.arange(K)[None, :, None] * 0.1
valid = torch.ones(B, K, dtype=torch.bool); valid[:12, -1] = False
class ChunkCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.posterior = nn.Sequential(nn.Linear(3 + K*A, 32), nn.ReLU(), nn.Linear(32, 2*Z))
        self.decoder = nn.Sequential(nn.Linear(3 + Z, 32), nn.ReLU(), nn.Linear(32, K*A))
    def decode(self, observation, z):
        return self.decoder(torch.cat([observation, z], -1)).reshape(-1, K, A)
    def forward(self, observation, targets, mask):
        # Zero padded targets before feeding the posterior as well as masking loss.
        encoded = torch.cat([observation, (targets * mask[..., None]).flatten(1)], -1)
        mu, logvar = self.posterior(encoded).chunk(2, -1)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
        return self.decode(observation, z), mu, logvar
model = ChunkCVAE()
assert model.decode(state, torch.zeros(B, Z)).shape == actions.shape

## Step 2

Verify zero KL for a standard-normal posterior, then train the masked objective.

In [ ]:
def kl_normal(mu, logvar):
    return 0.5 * (mu.square() + logvar.exp() - 1 - logvar).sum(-1).mean()
assert kl_normal(torch.zeros(B, Z), torch.zeros(B, Z)) == 0
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
history = []
for step in range(100):
    prediction, mu, logvar = model(state, actions, valid)
    reconstruction = ((prediction - actions).abs() * valid[..., None]).sum() / (valid.sum() * A)
    loss = reconstruction + 0.01 * kl_normal(mu, logvar)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
with torch.no_grad():
    predicted = model.decode(state, torch.zeros(B, Z))
print('z=0 inference L1 on train:', (predicted - actions).abs().mean().item())
assert torch.isfinite(predicted).all()
plt.plot(history, color='#d73535'); plt.xlabel('step'); plt.ylabel('L1 + beta KL'); plt.show()

## Step 3

At t=2 combine a[0,2], a[1,1], a[2,0]. A zero action remains a valid prediction, not missing data.

In [ ]:
chunks = torch.tensor([[[0., 0.], [1., 1.], [2., 2.], [3., 3.]],
                       [[10., 10.], [11., 11.], [12., 12.], [13., 13.]],
                       [[20., 20.], [21., 21.], [22., 22.], [23., 23.]]])
now = 2; starts = torch.arange(3)
candidates = chunks[starts, now - starts]
ages = (now - starts).float(); weights = torch.exp(-0.5 * ages); weights /= weights.sum()
ensemble = (candidates * weights[:, None]).sum(0)
print('same-time candidates:', candidates.tolist(), 'weights:', weights.tolist(), 'ensemble:', ensemble.tolist())
assert torch.equal(candidates[:, 0], torch.tensor([2., 11., 20.]))
assert ensemble.shape == (A,) and torch.allclose(weights.sum(), torch.tensor(1.))

## Reading the result

KL regularizes the posterior but does not guarantee useful latents; excessive beta can cause posterior collapse. Low target-conditioned reconstruction loss is not equivalent to good z=0 inference. Ensembling requires explicit timestamps and validity masks.

## Independent experiments

1. Increase beta 100-fold and track KL/reconstruction separately.
2. Supply actions at inference: why is that leakage?
3. Remove an old chunk and recompute the ensemble. Do not use action==0 as validity.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Large beta pushes the posterior toward the prior and may remove latent information. Deployment has no target actions. Chunk validity depends on prediction availability and timestamps, not the numerical action value.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.